In [4]:
from langchain.chat_models import ChatOpenAI
from langchain.agents import initialize_agent, AgentType, load_tools   
from langchain.tools import StructuredTool, Tool, BaseTool
from pydantic import BaseModel, Field
from langchain.tools import BaseTool
from typing import Any, Type
from langchain.utilities import DuckDuckGoSearchAPIWrapper
import os, requests

llm = ChatOpenAI(
    temperature=0.1,
)


In [3]:

alpha_vantage_api_key = os.environ.get("ALPHA_VANTAGE_API_KEY")

class stockMarketSymbolSearchToolArgsSchema(BaseModel):
    query: str = Field(description="The query you will search for")

class stockMarketSymbolSearchTool(BaseTool):
    name = "stockMarketSymbolSearchTool"
    description = """
    Use this tool to find the stock market symbol for a company.
    It takes a query as an argument.
    Example query : Stock Market Symbol for Apple Company
    """

    args_schema : Type[stockMarketSymbolSearchToolArgsSchema] = stockMarketSymbolSearchToolArgsSchema

    def _run(self, query):
        ddg = DuckDuckGoSearchAPIWrapper()
        return ddg.run(query)
    
class CompanyOverviewSchema(BaseModel):
    symbol: str = Field(description="Symbol of the company. Example: APPL, TSLA")

class CompanyOverviewTool(BaseTool):
    name = "CompanyOverview"
    description = "Use this tool to get an overview of the financials of the company. You should enter a stock symbol."
    args_schema : Type[CompanyOverviewSchema] = CompanyOverviewSchema
    
    def _run(self, symbol):
        ddg = DuckDuckGoSearchAPIWrapper()
        r = requests.get("https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={alpha_vantage_api_key}")
        return r.json()

class CompanyIncomeStatementTool(BaseTool):
    name = "CompanyIncomeStatement"
    description = "Use this tool to get an income state of the financials of the company. You should enter a stock symbol."
    args_schema : Type[CompanyOverviewSchema] = CompanyOverviewSchema
    
    def _run(self, symbol):
        ddg = DuckDuckGoSearchAPIWrapper()
        r = requests.get("https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={symbol}&apikey={alpha_vantage_api_key}")
        return r.json()
    
class CompanyStockPerformanceTool(BaseTool):
    name = "CompanyStockPerformance"
    description = "Use this tool to get the weekly performance of a company stock. You should enter a stock symbol."
    args_schema : Type[CompanyOverviewSchema] = CompanyOverviewSchema
    
    def _run(self, symbol):
        ddg = DuckDuckGoSearchAPIWrapper()
        r = requests.get("https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={alpha_vantage_api_key}")
        return r.json()
    

prompt = """
    Give me information on Cloudfare's stock, considering its financials and income statements and help me analyze if it's a potential good investment.
    """

agent = initialize_agent(
    llm = llm,
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    tools=[
        stockMarketSymbolSearchTool(),
        CompanyOverviewTool(),
        CompanyIncomeStatementTool(),
        CompanyStockPerformanceTool()
    ]
)

agent.invoke(prompt)
        



> Entering new AgentExecutor chain...

Invoking: `stockMarketSymbolSearchTool` with `{'query': 'Stock Market Symbol for Cloudfare'}`


Stock analysis for Cloudflare Inc (NET:New York) including stock price, stock chart, company news, key statistics, fundamentals and company profile. Cloudflare, Inc. (NYSE:NET) announced its quarterly earnings data on Thursday, August, 1st. The company reported $0.20 EPS for the quarter, topping analysts' consensus estimates of $0.14 by $0.06. The business's revenue for the quarter was up 30.0% compared to the same quarter last year. Real-time Cloudflare stock price (NYSE: NET) plus other valuable data points like day range, year, stock analyst insights, related news and more. RE. Cloudflare, Inc. Reports Earnings Results for the Second Quarter and Six Months Ended June 30, 2024. Aug. 01. CI. Earnings Flash (NET) CLOUDFLARE Reports Q2 Revenue $401M, vs. Street Est of $394.5M. Aug. 01. MT. Earnings Flash (NET) CLOUDFLARE Posts Q2 EPS $0.20, vs. Street 

{'input': "\n    Give me information on Cloudfare's stock, considering its financials and income statements and help me analyze if it's a potential good investment.\n    ",
 'output': "It seems that I've reached the rate limit for retrieving the company overview for Cloudfare. I'll need to wait before I can fetch the information."}

In [1]:
from langchain.agents import create_sql_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.sql_database import SQLDatabase

## db연결
db = SQLDatabase.from_uri("sqlite:///movies.sqlite")

In [5]:

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

agent_kit = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    #handle_parsing_error=True,
    verbose=True, 
)

agent_kit.invoke("Give me the 5 directors that have the highest grossing films")




> Entering new AgentExecutor chain...

Invoking: `sql_db_list_tables` with ``


directors, movies
Invoking: `sql_db_schema` with `directors, movies`



CREATE TABLE directors (
	name TEXT, 
	id INTEGER, 
	gender INTEGER, 
	uid INTEGER, 
	department TEXT, 
	PRIMARY KEY (id)
)

/*
3 rows from directors table:
name	id	gender	uid	department
James Cameron	4762	2	2710	Directing
Gore Verbinski	4763	2	1704	Directing
Sam Mendes	4764	2	39	Directing
*/


CREATE TABLE movies (
	id INTEGER, 
	original_title VARCHAR, 
	budget INTEGER, 
	popularity INTEGER, 
	release_date TEXT, 
	revenue INTEGER, 
	title TEXT, 
	vote_average REAL, 
	vote_count INTEGER, 
	overview TEXT, 
	tagline TEXT, 
	uid INTEGER, 
	director_id INTEGER DEFAULT 0 NOT NULL, 
	PRIMARY KEY (id)
)

/*
3 rows from movies table:
id	original_title	budget	popularity	release_date	revenue	title	vote_average	vote_count	overview	tagline	uid	director_id
43597	Avatar	237000000	150	2009-12-10	2787965087	Avatar	7.2	11800	In the 22nd century, a pa

{'input': 'Give me the 5 directors that have the highest grossing films',
 'output': 'The 5 directors with the highest grossing films are:\n\n1. Steven Spielberg - $9,147,393,164\n2. Peter Jackson - $6,498,642,820\n3. James Cameron - $5,883,569,439\n4. Michael Bay - $5,832,524,638\n5. Christopher Nolan - $4,227,483,234'}